# Chapter 8: Deploying a GenAI Application with MLflow

This notebook takes the Unity Airways booking agent from a registered model to a governed production deployment: logging and evaluating it, serving it on Databricks Model Serving, and fronting it with the MLflow AI Gateway.

## 📓 About this notebook
This notebook takes the Unity Airways booking agent from a registered model to a governed production deployment: logging and evaluating it, serving it on Databricks Model Serving, and fronting it with the MLflow AI Gateway.

**Maps to the book:** Chapter 8, *Deploying a GenAI Application with MLflow* — sections: Deploying to Databricks Model Serving, MLflow AI Gateway, AI Guardrails, LLM Operations and Agent Operations, Challenges of Productionization.

### ✅ Prerequisites

Run these before this notebook:

1. [`Appendix/data_ingestion`](../Appendix/data_ingestion) — creates the tables and FAQ **vector search index** the agent's tools depend on.
2. [Chapter 7](../Chapter07) — builds the `ResponsesAgent` and its Unity Catalog tools that this chapter deploys.

Deploying to Model Serving requires permission to create serving endpoints. See the [repository README](../README.md).

In [0]:
%pip install -r ../requirements.txt
dbutils.library.restartPython()


## Define the agent in code
Define the agent code in a single cell below. This lets you easily write the agent code to a local Python file, using the `%%writefile` magic command, for subsequent logging and deployment.




### 1. Wrap the LangGraph agent using the `ResponsesAgent` interface

For compatibility with Databricks AI features, the `LangGraphResponsesAgent` class implements the `ResponsesAgent` interface to wrap the LangGraph agent.

Databricks recommends using `ResponsesAgent` as it simplifies authoring multi-turn conversational agents using an open source standard. See MLflow's [ResponsesAgent documentation](https://www.mlflow.org/docs/latest/llms/responses-agent-intro/).


In [0]:
# %%writefile agent.py
import json
from typing import Annotated, Any, Generator, Optional, Sequence, TypedDict, Union, Dict
from uuid import uuid4
import openmeteo_requests
import mlflow
from databricks_langchain import (
    ChatDatabricks,
    UCFunctionToolkit,
    VectorSearchRetrieverTool,
)
from langchain_core.language_models import LanguageModelLike
from langchain_core.messages import (
    AIMessage,
    AIMessageChunk,
    BaseMessage,
    convert_to_openai_messages,
)
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langchain_core.tools import BaseTool, StructuredTool
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from mlflow.entities import SpanType
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
)
import pandas as pd
from pydantic import BaseModel
from langgraph.prebuilt import ToolNode

############################################
# Define your LLM endpoint and system prompt
############################################
LLM_ENDPOINT_NAME = "databricks-meta-llama-3-3-70b-instruct"
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)

system_prompt = """You are an helpful flight assistant for Unity Airways, capable of supporting users with their bookings and policy questions using tools to retrieve information from Unity Airways knowledge bases.

Available user requests you can handle:
	•	Help customers find their flight reservations using booking reference (PNR), last name with email or phone, or ticket number.
	•	Assist in changing flight dates/times or canceling bookings, calculating and informing users about any applicable fees or waivers, and processing cancellations or changes.
	•	Answer customer questions about baggage allowances, change and refund rules, or disruption waivers by searching the policy FAQ.

You have access to these tools:
	•	lookup_booking: Given a PNR, ticket number, or name plus email/phone, retrieve full booking and policy context.
	•	modify_cancel_booking: Given booking context, change or cancel bookings, determine fees/waivers, and update booking state.
	•	faq_search: Given a natural language policy question, return the best answer, its source, and a confidence score.

When a customer asks a question:
	1.	Identify the primary intent: find booking, modify/cancel, or a policy query.
	2.	Gather necessary information (prompt the customer for any missing details).
	3.	Use the correct tool(s) to answer the request or perform the action.
	4.	For booking or change/cancel actions, clearly explain the outcome, including eligibility, rules, fees, refunds, or waivers.
	5.	If a policy query is made, search the FAQ and provide the most relevant, accurate information with attribution.
	6.	Escalate to a human agent if needed.
 
Always remain friendly, concise, and clear. Always confirm required information before taking any action. Make sure your responses are personalized and relevant to the user's intent. If the action asked cannot be handled, politely say so, and redirect to customer service contact email: support@unityairways.com. """

###############################################################################
## Define tools for your agent, enabling it to retrieve data or take actions
## To create and see usage examples of more tools, see https://docs.databricks.com/en/generative-ai/agent-framework/agent-tool.html
###############################################################################
tools = []

# You can use UDFs in Unity Catalog as agent tools
UC_TOOL_NAMES = ["workspace.unity_airways.lookup_customer_info", "workspace.unity_airways.modify_cancel_booking"]
uc_toolkit = UCFunctionToolkit(function_names=UC_TOOL_NAMES)
tools.extend(uc_toolkit.tools)

#############################
## Vector Search for FAQ Tool
#############################
# Use Databricks vector search indexes as tools
VECTOR_SEARCH_TOOLS = []

VECTOR_SEARCH_TOOLS.append(
    VectorSearchRetrieverTool(
        index_name="workspace.unity_airways.faq_index",
        num_results=5,
        tool_description="Search through unity airways frequently asked question (FAQ) about flight cancellation, travel policies and baggages security."
    )
)
tools.extend(VECTOR_SEARCH_TOOLS)
#####################
## Define agent logic
#####################

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    custom_inputs: Optional[dict[str, Any]]
    custom_outputs: Optional[dict[str, Any]]

def create_tool_calling_agent(
    model: LanguageModelLike,
    tools: Union[ToolNode, Sequence[BaseTool]],
    system_prompt: Optional[str] = None,
):
    print(tools)
    model = model.bind_tools(tools)

    # Define the function that determines which node to go to
    def should_continue(state: AgentState):
        messages = state["messages"]
        last_message = messages[-1]
        # If there are function calls, continue. else, end
        if isinstance(last_message, AIMessage) and last_message.tool_calls:
            return "continue"
        else:
            return "end"

    if system_prompt:
        preprocessor = RunnableLambda(
            lambda state: [{"role": "system", "content": system_prompt}] + state["messages"]
        )
    else:
        preprocessor = RunnableLambda(lambda state: state["messages"])
    model_runnable = preprocessor | model

    def call_model(
        state: AgentState,
        config: RunnableConfig,
    ):
        response = model_runnable.invoke(state, config)

        return {"messages": [response]}

    workflow = StateGraph(AgentState)

    workflow.add_node("agent", RunnableLambda(call_model))
    workflow.add_node("tools", ToolNode(tools))
    workflow.set_entry_point("agent")
    workflow.add_conditional_edges(
        "agent",
        should_continue,
        {
            "continue": "tools",
            "end": END,
        },
    )
    workflow.add_edge("tools", "agent")

    return workflow.compile()


class LangGraphResponsesAgent(ResponsesAgent):
    def __init__(self, agent):
        self.agent = agent

    def _responses_to_cc(self, message: dict[str, Any]) -> list[dict[str, Any]]:
        """Convert from a Responses API output item to ChatCompletion messages."""
        msg_type = message.get("type")
        if msg_type == "function_call":
            return [
                {
                    "role": "assistant",
                    "content": "tool call",
                    "tool_calls": [
                        {
                            "id": message["call_id"],
                            "type": "function",
                            "function": {
                                "arguments": message["arguments"],
                                "name": message["name"],
                            },
                        }
                    ],
                }
            ]
        elif msg_type == "message" and isinstance(message["content"], list):
            return [
                {"role": message["role"], "content": content["text"]}
                for content in message["content"]
            ]
        elif msg_type == "reasoning":
            return [{"role": "assistant", "content": json.dumps(message["summary"])}]
        elif msg_type == "function_call_output":
            return [
                {
                    "role": "tool",
                    "content": message["output"],
                    "tool_call_id": message["call_id"],
                }
            ]
        compatible_keys = ["role", "content", "name", "tool_calls", "tool_call_id"]
        filtered = {k: v for k, v in message.items() if k in compatible_keys}
        return [filtered] if filtered else []

    def _prep_msgs_for_cc_llm(self, responses_input) -> list[dict[str, Any]]:
        "Convert from Responses input items to ChatCompletion dictionaries"
        cc_msgs = []
        for msg in responses_input:
            cc_msgs.extend(self._responses_to_cc(msg.model_dump()))

    def _langchain_to_responses(self, messages: list[dict[str, Any]]) -> list[dict[str, Any]]:
        "Convert from ChatCompletion dict to Responses output item dictionaries"
        for message in messages:
            message = message.model_dump()
            role = message["type"]
            if role == "ai":
                if tool_calls := message.get("tool_calls"):
                    return [
                        self.create_function_call_item(
                            id=message.get("id") or str(uuid4()),
                            call_id=tool_call["id"],
                            name=tool_call["name"],
                            arguments=json.dumps(tool_call["args"]),
                        )
                        for tool_call in tool_calls
                    ]
                else:
                    return [
                        self.create_text_output_item(
                            text=message["content"],
                            id=message.get("id") or str(uuid4()),
                        )
                    ]
            elif role == "tool":
                return [
                    self.create_function_call_output_item(
                        call_id=message["tool_call_id"],
                        output=message["content"],
                    )
                ]
            elif role == "user":
                return [message]

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        outputs = [
            event.item
            for event in self.predict_stream(request)
            if event.type == "response.output_item.done"
        ]
        return ResponsesAgentResponse(output=outputs, custom_outputs=request.custom_inputs)

    def predict_stream(
        self,
        request: ResponsesAgentRequest,
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        cc_msgs = []
        for msg in request.input:
            cc_msgs.extend(self._responses_to_cc(msg.model_dump()))

        for event in self.agent.stream({"messages": cc_msgs}, stream_mode=["updates", "messages"]):
            if event[0] == "updates":
                for node_data in event[1].values():
                    for item in self._langchain_to_responses(node_data["messages"]):
                        yield ResponsesAgentStreamEvent(type="response.output_item.done", item=item)
            # filter the streamed messages to just the generated text messages
            elif event[0] == "messages":
                try:
                    chunk = event[1][0]
                    if isinstance(chunk, AIMessageChunk) and (content := chunk.content):
                        yield ResponsesAgentStreamEvent(
                            **self.create_text_delta(delta=content, item_id=chunk.id),
                        )
                except Exception as e:
                    print(e)


# Create the agent object, and specify it as the agent object to use when
# loading the agent back for inference via mlflow.models.set_model()
mlflow.langchain.autolog()
agent = create_tool_calling_agent(llm, tools, system_prompt)
response_agent = LangGraphResponsesAgent(agent)
mlflow.models.set_model(response_agent)

In [0]:
# from agent import agent
result = agent.invoke({"messages": [{"role": "user", "content": "can the booking c3dd03uvulemx15z be cancelled?"}]})
result

In [0]:
from IPython.display import Image, display

# Comment in the agent code `%%writefile agent.py` if you want to run the graph visualization
display(Image(agent.get_graph().draw_mermaid_png()))

### 2. Test the agent

Interact with the agent to test its output and tool-calling abilities. Since this notebook called `mlflow.langchain.autolog()`, you can view the trace for each step the agent takes.

Replace this placeholder input with an appropriate domain-specific example for your agent.

In [0]:
from agent import response_agent
result = response_agent.predict({"input": [{"role": "user", "content": "can the booking c3dd03uvulemx15z be cancelled?"}]})
result

In [0]:
result = response_agent.predict({"input": [{"role": "user", "content": "what is the weather in singapore for the upcoming week?"}]})
result

In [0]:
for chunk in response_agent.predict_stream({"input": [{"role": "user", "content": "what's the policy for rescheduling my flight?"}]}):
    print(chunk.model_dump(exclude_none=True))

### 3. Log the agent as an MLflow model

Log the agent as code from the `agent.py` file. See [MLflow - Models from Code](https://mlflow.org/docs/latest/models.html#models-from-code).

### Enable automatic authentication for Databricks resources
For the most common Databricks resource types, Databricks supports and recommends declaring resource dependencies for the agent upfront during logging. This enables automatic authentication passthrough when you deploy the agent. With automatic authentication passthrough, Databricks automatically provisions, rotates, and manages short-lived credentials to securely access these resource dependencies from within the agent endpoint.

To enable automatic authentication, specify the dependent Databricks resources when calling `mlflow.pyfunc.log_model().`

  - **Note:** If your Unity Catalog tool queries a vector search index or leverages external functions, you need to include the dependent vector search index and UC connection objects, respectively, as resources. See docs ([AWS](https://docs.databricks.com/generative-ai/agent-framework/log-agent.html#specify-resources-for-automatic-authentication-passthrough) | [Azure](https://learn.microsoft.com/azure/databricks/generative-ai/agent-framework/log-agent#resources)).



In [0]:
# Determine Databricks resources to specify for automatic auth passthrough at deployment time
from agent import UC_TOOL_NAMES, VECTOR_SEARCH_TOOLS
import mlflow
from mlflow.models.resources import DatabricksFunction
from pkg_resources import get_distribution

resources = []
for tool in VECTOR_SEARCH_TOOLS:
    resources.extend(tool.resources)
for tool_name in UC_TOOL_NAMES:
    resources.append(DatabricksFunction(function_name=tool_name))

with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model="agent.py",
        pip_requirements=[
            "databricks-langchain",
            f"langgraph=={get_distribution('langgraph').version}",
            f"backoff=={get_distribution('backoff').version}",
            f"databricks-connect=={get_distribution('databricks-connect').version}",
            f"openmeteo_requests=={get_distribution('openmeteo_requests').version}",
        ],
        resources=resources,
    )

## Evaluate the tools with Agent Evaluation

Use Mosaic AI Agent Evaluation to evaluate the agent's responses based on expected responses and other evaluation criteria. Use the evaluation criteria you specify to guide iterations, using MLflow to track the computed quality metrics.
See Databricks documentation ([AWS](https://docs.databricks.com/aws/generative-ai/agent-evaluation) | [Azure](https://learn.microsoft.com/azure/databricks/generative-ai/agent-evaluation/)).


To evaluate your tool calls, add custom metrics. See Databricks documentation ([AWS](https://docs.databricks.com/en/generative-ai/agent-evaluation/custom-metrics.html#evaluating-tool-calls) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-evaluation/custom-metrics#evaluating-tool-calls)).

In [0]:
import mlflow
from mlflow.genai.scorers import RelevanceToQuery, RetrievalGroundedness, RetrievalRelevance, Safety

eval_dataset = [
    {
        "inputs": {"input": [{"role": "user", "content": "Can I bring my battery to cabin luggage?"}]},
        "expected_response": "Spare lithium batteries and power banks can be in carry-on only with terminals protected. Size limits apply per IATA guidance.",
    }
]

eval_results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=lambda input: response_agent.predict({"input": input}),
    scorers=[RelevanceToQuery(), Safety()],  # add more scorers here if they're applicable
)

# Review the evaluation results in the MLfLow UI (see console output)

## Prepare for Deployment
Before registering and deploying the agent, perform pre-deployment checks using the [mlflow.models.predict()](https://mlflow.org/docs/latest/python_api/mlflow.models.html#mlflow.models.predict) API. See Databricks documentation ([AWS](https://docs.databricks.com/en/machine-learning/model-serving/model-serving-debug.html#validate-inputs) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/machine-learning/model-serving/model-serving-debug#before-model-deployment-validation-checks)).

### 1. Pre-deployment agent validation
Before registering and deploying the agent, perform pre-deployment checks using the [mlflow.models.predict()](https://mlflow.org/docs/latest/python_api/mlflow.models.html#mlflow.models.predict) API. See Databricks documentation ([AWS](https://docs.databricks.com/en/machine-learning/model-serving/model-serving-debug.html#validate-inputs) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/machine-learning/model-serving/model-serving-debug#before-model-deployment-validation-checks)).

In [0]:
mlflow.models.predict(
    model_uri=f"runs:/{logged_agent_info.run_id}/agent",
    input_data={"input": [{"role": "user", "content": "Can my battery pack be transported in cabin baggages in this weather in singapore?"}]},
    env_manager="uv",
)

### 2. Register the model to Unity Catalog

Before you deploy the agent, you must register the agent to Unity Catalog.


In [0]:
mlflow.set_registry_uri("databricks-uc")

catalog = "workspace"
schema = "unity_airways"
model_name = "unity-airways-booking-agent"
UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

# register the model to UC
uc_registered_model_info = mlflow.register_model(model_uri=logged_agent_info.model_uri, name=UC_MODEL_NAME)

### Deploy to Databricks Model Serving
`agents.deploy` provisions a Model Serving endpoint for the agent (scale-to-zero keeps idle cost low) plus a companion Review App where SMEs can chat with it and leave feedback. _(see Ch 8, "Deploying to Databricks Model Serving")_

### 3. Deploy the agent

In [0]:
from databricks import agents

agents.deploy(
    model_name=UC_MODEL_NAME,
    model_version=uc_registered_model_info.version,
    scale_to_zero=True
)

### Call the served endpoint
With the endpoint live, downstream applications reach the agent over HTTP; here we stream a multi-turn conversation and ask for the trace back for observability. _(see Ch 8, "Model Serving Endpoint")_

### Front the endpoint with MLflow AI Gateway
The Unity AI Gateway wraps MLflow AI Gateway as a central proxy in front of model endpoints, adding access control, rate limiting, and a place to attach AI guardrails. _(see Ch 8, "MLflow AI Gateway", "AI Guardrails")_

In [0]:
import requests
import json

workspace_serving_url = "https://dbc-5c83b309-9456.cloud.databricks.com/serving-endpoints/responses"
model_name = "agents_workspace-unity_airways-unity-airways-booking-agent"
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}

payload = {
    "model": model_name,
    "input": [{"role": "user", "content": "what are the risks of travelling with my friends luggage?"},
      {"role": "assistant", "content": "It is a good practice to not travel with a luggage and verify they have not been tampered with before you check them in."},
        {"role": "user", "content": "hi, how many kg for cabin baggages i can bring?"}
    ],
    "stream": True,
    "custom_inputs": { "id": 5 },
    "databricks_options": { "return_trace": True }
}

with requests.post(workspace_serving_url, headers=headers, json=payload, stream=True) as response:
    for line in response.iter_lines():
        if line:
            decoded_line = line.decode("utf-8")
            print(decoded_line)

### Agent on App Deployment
https://docs.databricks.com/aws/en/generative-ai/agent-framework/migrate-agent-to-apps

### Route requests through the gateway
Pointing the OpenAI client at the gateway URL sends traffic through the governed proxy instead of the raw model endpoint, applying the rate limits and guardrails configured there — a foundation for the LLMOps/AgentOps practices (challenger vs. champion, staged rollouts) discussed at the end of the chapter. _(see Ch 8, "MLflow AI Gateway", "LLM Operations and Agent Operations")_

In [0]:
from databricks_langchain import ChatDatabricks


LLM_ENDPOINT_NAME = "databricks-meta-llama-3-3-70b-instruct"
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)


In [0]:
from openai import OpenAI
import os

DATABRICKS_TOKEN = os.environ.get('DATABRICKS_TOKEN')

client = OpenAI(
  api_key=DATABRICKS_TOKEN,
  base_url="https://<ai-gateway-url>/openai/v1"
)

response = client.responses.create(
  model="<ai-gateway-endpoint>",
  max_output_tokens=256,
  input=[
    {
      "role": "user",
      "content": [{"type": "input_text", "text": "What is Databricks?"}]
    }
  ]
)

print(response.output)

In [0]:
from openai import OpenAI                                                                                                 
                                                                                                                        
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()                                
                                                                                                                        
client = OpenAI(
  base_url="https://dbc-5c83b309-9456.cloud.databricks.com.databricks.com/serving-endpoints",
  api_key=token
)

response = client.chat.completions.create(
  model="databricks-meta-llama-3-3-70b-instruct",
  messages=[{"role": "user", "content": "Hello!"}],
)

print(response.choices[0].message.content)

In [0]:
from openai import OpenAI
import os
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

client = OpenAI(
    base_url="https://890276512801139.ai-gateway.cloud.databricks.com/mlflow/v1",
    api_key=token
)

response = client.chat.completions.create(
    model="unity_gateway",
    messages=[{"role": "user", "content": "Hello! How are you doing"}],
)

print(response.choices[0].message.content)

In [0]:
llm = ChatDatabricks(
    endpoint="unity_gateway",
    target_uri="https://890276512801139.ai-gateway.cloud.databricks.com",
)



## Next steps

After your agent is deployed, you can chat with it in AI playground to perform additional checks, share it with SMEs in your organization for feedback, or embed it in a production application. See docs ([AWS](https://docs.databricks.com/en/generative-ai/deploy-agent.html) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/deploy-agent)) for details